In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

In [ ]:
np.set_printoptions(threshold=sys.maxsize)

# Dropout Layer

Dropout acts as a _regularization_ technique that removes or deactivates nodes during the forward pass, which makes the network less prone to overfitting by keeping it from relying too heavily on any single neuron. This layer has no parameters.

* In the forward pass, inputs are set to zero with probability $p$, and otherwise scaled by $\frac{1}{1-p}$.

  - The forward pass during training only serves to set the network up for the backward pass, which is where the network is actually modified.
  
  - For each individual neuron in the layer we can say that $x \sim B(1, p)$, since we are considering a single "experiment" (the neuron being switched on or off) with a success probability of $p$.

* In the backward pass, the gradients of those same dropped units are zeroed out; the remaining gradients are scaled by the same factor of $\frac{1}{1-p}$.

  - In other words, if a node was dropped by the Dropout layer, then its influence (the gradient) on the outgoing weights is 0 as well (since $0 * w_i = 0$). In short, backpropagation works just as it always does.


> NOTE: keep in mind that during the test or validation phase every neuron is active (that is, no Dropout is applied), so that the prediction is based on the whole network.

In [ ]:
np.random.seed(123)

din=10
batch_size=2

x = np.random.rand(batch_size,din)
                       
layer=nn.Dropout(p=0.5)

In [ ]:
x

In [ ]:
y = layer.forward(x)
y, y.shape

In [ ]:
# Define the gradient of the output
g = np.random.rand(*y.shape)

# Propagate the gradient backward through the convolution
layer_grad = layer.backward(g)
layer_grad

In [ ]:
from edunn.model import Phase
layer.set_phase(Phase.Test)
y = layer.forward(x)
y, y.shape

## Checks

In [ ]:
x = np.ones((100,100))

layer=nn.Dropout(p=0.2)
y = layer.forward(x)
utils.check_same_float(0.2,(y==0).mean(),"Fraction of zeros",tol=0.02)
utils.check_same(x[y>0]/0.8,y[y>0])

g = np.random.rand(*y.shape)
utils.check_same(g*y,layer.backward(g)[0])

layer.set_phase(Phase.Test)
utils.check_same(x,layer.forward(x))